In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import seaborn as sns  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.filtering.dataset_filter import DatasetFilter  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
    MusicTypeVariants,
    ConditionVariants,
    ExclusionCategories,
)
from src.visualization.preprocessing_plots import DatasetPlotter  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
print("Setup complete.")

# Time Alignment Analysis

This notebook analyses and verifies the **time alignment** of EEG recordings across
participants using the TAG (stimulus marker) channel.

Capabilities:
* Run cross-correlation-based time alignment across participants
* Plot cross-correlation curves for each participant pair vs. the reference
* Overlay all TAG music signals to verify alignment
* Compare aligned vs. unaligned signals visually
* Plot the pairwise correlation heatmap of aligned TAG signals

> **Parameters to tweak:** `CONDITION`, `MUSIC_TYPE`, `EXCLUSION_CATEGORIES`,
> `PLOT_WINDOW_SEC`, `CROSSCORR_MAX_LAG_SEC` in the *Configuration* cell below.

## Configuration

In [ ]:
# ── Experiment selection ──────────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC]

# ── Time alignment parameters ─────────────────────────────────────────────────
# Set to True to actually crop and save aligned data after alignment
CROP_AND_SAVE = False

# ── Visualisation parameters ──────────────────────────────────────────────────
PLOT_WINDOW_SEC = 10.0  # seconds of signal to show in the overlap plot
PLOT_T_START_SEC = 0.0  # start time (s) for the overlap plot
CROSSCORR_MAX_LAG_SEC = 5.0  # ±lag range (s) for the cross-correlation plots

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = ProjectPaths.NOTEBOOKS_DIR / "00-preprocessing" / "plots" / "time_alignment"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
_plot_suffix = f"{CONDITION.value}_{MUSIC_TYPE.value}"
print(f"Plots will be saved to: {PLOTS_DIR}")

## Dataset Initialisation

In [ ]:
dataset_handler = DatasetHandler(
    ExperimentNames.PSILO_MUSIC, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
dataset_metadata = dataset_handler.dataset_metadata
print(f"Dataset size: {len(dataset_metadata)} recordings")

## Run Time Alignment

Align all recordings for the selected condition and music type using cross-correlation
of the TAG (stimulus marker) signals.

In [ ]:
aligned_signals, time_aligner = dataset_handler.align_time_series(
    MUSIC_TYPE,
    CONDITION,
    EXCLUSION_CATEGORIES,
    plot_alignment_results=False,
    data_type_to_load=PreprocessedDataVariants.RAW_AFTER_ICA,
)
print(f"Aligned {len(aligned_signals)} recordings")
if time_aligner is not None:
    print(f"Reference index: {time_aligner.reference_index}")
    print(f"Shifts (samples): {time_aligner.shifts}")

## Cross-Correlation Plots

Plot the cross-correlation function for each recording vs. the reference to inspect
the quality and uniqueness of the alignment peak.

In [ ]:
if aligned_signals and time_aligner is not None:
    ref_idx = time_aligner.reference_index
    ref_signal = aligned_signals[ref_idx]
    sfreq = time_aligner.sfreq

    for i, sig in enumerate(aligned_signals):
        if i == ref_idx:
            continue
        save_path = (
            str(PLOTS_DIR / f"crosscorr_{_plot_suffix}_vs_ref_idx{i}.png")
            if SAVE_PLOTS
            else ""
        )
        DatasetPlotter.plot_crosscorr_vs_shift(
            ref_signal,
            sig,
            sfreq=sfreq,
            max_lag_sec=CROSSCORR_MAX_LAG_SEC,
            label1=f"Reference (idx {ref_idx})",
            label2=f"Recording {i}",
            save_fig=save_path,
        )
else:
    print("No aligned signals available — check configuration.")

## Aligned TAG Signal Overlay

After alignment, load the cropped TAG signals and plot all overlaid on the same axis
to verify synchrony.

In [ ]:
filtered_df = DatasetFilter.filter_dataset_by_all_categories(
    dataset_handler.dataset_metadata,
    dataset_handler.excluded_participants_metadata,
    [MUSIC_TYPE],
    [CONDITION],
    EXCLUSION_CATEGORIES,
)

all_tags, sfreq = dataset_handler._load_all_tags(
    filtered_df, PreprocessedDataVariants.RAW_AFTER_ICA
)
tag_signals = [tag.tag_signal for tag in all_tags]
print(f"Loaded {len(tag_signals)} TAG signals, sampling rate: {sfreq} Hz")

In [ ]:
# Plot the aligned TAG signals via DatasetHandler helper
dataset_handler.plot_aligned_tags(tag_signals, time_aligner)

## Signal Overlap Plot

Show a short time window of all aligned TAG signals overlaid for detailed inspection.

In [ ]:
if tag_signals:
    # After alignment, derive the aligned (shifted) signals for overlay
    if time_aligner is not None:
        shifts = time_aligner.shifts
        max_shift = max(abs(s) for s in shifts)
        aligned_tags = [
            sig[max_shift + s : max_shift + s + (len(sig) - 2 * max_shift)]
            for sig, s in zip(tag_signals, shifts)
        ]
    else:
        aligned_tags = tag_signals

    DatasetPlotter.plot_signal_overlap(
        aligned_tags,
        sfreq=sfreq,
        t_start=PLOT_T_START_SEC,
        time_duration=PLOT_WINDOW_SEC,
        save_fig=str(PLOTS_DIR / f"signal_overlap_{_plot_suffix}.png")
        if SAVE_PLOTS
        else "",
    )
else:
    print("No TAG signals available.")

## Pairwise Correlation Heatmap

Compute and visualise pairwise Pearson correlations between all aligned TAG signals
to quantify how well the recordings are synchronised.

In [ ]:
if tag_signals:
    DatasetPlotter.print_correlation_statistics(tag_signals)
    DatasetPlotter.plot_alignment_correlation_heatmap(
        tag_signals,
        save_fig=str(PLOTS_DIR / f"correlation_heatmap_{_plot_suffix}.png")
        if SAVE_PLOTS
        else "",
    )
else:
    print("No TAG signals available.")

## Apply Alignment (Crop and Save)

Set `CROP_AND_SAVE = True` in the Configuration cell to crop all recordings to the
common aligned time window and save them to disk.

In [ ]:
if CROP_AND_SAVE:
    if time_aligner is None:
        raise RuntimeError("time_aligner is None — run the alignment cell first.")
    dataset_handler.crop_all_raw_to_alignment(time_aligner)
    print("All recordings cropped and saved.")
else:
    print("CROP_AND_SAVE is False — skipping. Set it to True to write files to disk.")